# Stage 3 · Modern RLHF Variants — EXERCISES
### Topics: DPO · IPO · SimPO · KTO · GRPO vs PPO · Offline vs Online Tradeoffs

> Fill every `# TODO`. Run `# ASSERT` cells to verify.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass


---
## 1 · Direct Preference Optimization (DPO)

**DPO (Rafailov et al., 2023)** is the key insight that made offline preference training practical.

### The key derivation
The RLHF objective with KL constraint has a closed-form optimal policy:
$$\pi^*(y|x) = \frac{1}{Z(x)}\pi_{ref}(y|x) \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

Rearranging: the reward can be expressed in terms of the policy and reference:
$$r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

Substituting into the Bradley-Terry objective and noting $Z(x)$ cancels:

$$\mathcal{L}_{DPO} = -\mathbb{E}\!\left[\log \sigma\!\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

### Why DPO is powerful
- **No reward model needed** — the policy directly parameterises the implicit reward
- **No RL loop** — standard supervised training on preference pairs
- **Stable** — no clipping, no GAE, no value network
- **Tradeoff** — offline, so distribution shift can be an issue; weaker than PPO on hard tasks


In [ ]:
def dpo_loss(
    logprobs_policy_chosen:   torch.Tensor,  # (B,)
    logprobs_policy_rejected: torch.Tensor,  # (B,)
    logprobs_ref_chosen:      torch.Tensor,  # (B,)  detached
    logprobs_ref_rejected:    torch.Tensor,  # (B,)  detached
    beta: float = 0.1,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Steps:
      log_ratio_w = logp_policy_chosen   - logp_ref_chosen.detach()
      log_ratio_l = logp_policy_rejected - logp_ref_rejected.detach()
      rewards_w = beta * log_ratio_w
      rewards_l = beta * log_ratio_l
      loss = -mean(log_sigmoid(rewards_w - rewards_l))
    Return (loss, dict with loss, reward_chosen, reward_rejected, reward_margin, accuracy).
    """
    # TODO
    raise NotImplementedError


def dpo_implicit_reward(
    logprobs_policy: torch.Tensor,  # (B,)
    logprobs_ref:    torch.Tensor,  # (B,)
    beta: float = 0.1,
) -> torch.Tensor:
    """Implicit reward: β * (log π_θ - log π_ref). Detach ref."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 8
lp_pw = torch.tensor([-1.0, -1.5, -0.8, -2.0, -1.2, -0.9, -1.8, -1.1])
lp_pl = torch.tensor([-2.0, -2.5, -1.8, -3.0, -2.2, -1.9, -2.8, -2.1])
lp_rw, lp_rl = lp_pw.clone(), lp_pl.clone()

loss_init, info_init = dpo_loss(lp_pw, lp_pl, lp_rw, lp_rl, beta=0.1)
assert abs(loss_init.item() - math.log(2)) < 1e-5,     f"policy==ref → loss should be log(2)={math.log(2):.4f}, got {loss_init.item():.4f}"

loss_opt, info_opt = dpo_loss(lp_rw + 1.0, lp_rl - 1.0, lp_rw, lp_rl, beta=0.1)
assert loss_opt.item() < loss_init.item() and info_opt["accuracy"] == 1.0
print(f"dpo_loss ✓  init={loss_init.item():.4f}, opt={loss_opt.item():.4f}, acc={info_opt['accuracy']:.2f}")


---
## 2 · DPO Variants: IPO, KTO, SimPO

Each variant fixes a known failure mode of DPO.

### IPO (Identity Preference Optimization, Azar et al., 2023)
DPO can overfit — the log-ratio can grow without bound if the model is unconstrained.  
IPO uses an **L2 regulariser** instead of log-sigmoid:

$$\mathcal{L}_{IPO} = \mathbb{E}\!\left[\left(\log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)} - \frac{1}{2\tau}\right)^2\right]$$

Target margin is $1/(2\tau)$; the policy is penalised for exceeding it.

### KTO (Kahneman-Tversky Optimization, Ethayarajh et al., 2024)
Works on **unpaired** data (just "good" or "bad" completions, no pairs).  
Uses prospect theory: losses feel larger than equivalent gains (loss aversion).

$$\mathcal{L}_{KTO} = \mathbb{E}\left[\lambda_w \cdot \sigma(\hat{r}_w - z_0) + \lambda_l \cdot \sigma(z_0 - \hat{r}_l)\right]$$

where $\hat{r} = \beta(\log \pi_\theta(y|x) - \log \pi_{ref}(y|x))$ and $z_0$ is an estimated KL baseline.

### SimPO (Simple Preference Optimization, Meng et al., 2024)
Eliminates the reference model entirely. Uses **length-normalised** log-probs and a **margin** γ:

$$\mathcal{L}_{SimPO} = -\mathbb{E}\!\left[\log \sigma\!\left(\frac{\beta}{|y_w|}\log \pi(y_w|x) - \frac{\beta}{|y_l|}\log \pi(y_l|x) - \gamma\right)\right]$$

Length normalisation prevents the model from preferring short completions.


In [ ]:
def ipo_loss(
    logprobs_policy_chosen:   torch.Tensor,  # (B,)
    logprobs_policy_rejected: torch.Tensor,  # (B,)
    logprobs_ref_chosen:      torch.Tensor,  # (B,)  detached
    logprobs_ref_rejected:    torch.Tensor,  # (B,)  detached
    tau: float = 0.1,
) -> torch.Tensor:
    """
    IPO: mean( (log_ratio_w - log_ratio_l - 1/(2τ))² )
    The policy is penalised for margin deviating from 1/(2τ).
    """
    # TODO
    raise NotImplementedError


def simpo_loss(
    logprobs_policy_chosen:   torch.Tensor,  # (B,)
    logprobs_policy_rejected: torch.Tensor,  # (B,)
    len_chosen:   torch.Tensor,              # (B,)  int — completion lengths
    len_rejected: torch.Tensor,              # (B,)  int
    beta:  float = 2.5,
    gamma: float = 0.5,
) -> Tuple[torch.Tensor, float]:
    """
    SimPO: no reference model, length-normalised.
    logit = β/|yw|*logπ(yw) - β/|yl|*logπ(yl) - γ
    loss  = -mean(log_sigmoid(logit))
    Return (loss, accuracy).
    """
    # TODO
    raise NotImplementedError


def kto_loss(
    logprobs_policy: torch.Tensor,   # (B,)
    logprobs_ref:    torch.Tensor,   # (B,)  detached
    is_good:         torch.Tensor,   # (B,) bool
    beta:  float = 0.1,
    lam_w: float = 1.0,
    lam_l: float = 1.0,
) -> torch.Tensor:
    """
    KTO: unpaired preference loss.
    r_hat = β * (logp_policy - logp_ref)
    z_0   = mean(r_hat).detach()
    loss  = -λ_w * mean(log_sigmoid(r_hat[good] - z_0))
            -λ_l * mean(log_sigmoid(z_0 - r_hat[bad]))
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 8
lp_rw = torch.randn(B)
lp_rl = torch.randn(B)

# IPO: loss==0 when log-ratio diff == 1/(2τ)
lp_pw_t = lp_rw + 1/(2*0.1)
ipo = ipo_loss(lp_pw_t, lp_rl, lp_rw, lp_rl, tau=0.1)
assert ipo.item() < 1e-8, f"Expected 0, got {ipo.item()}"

len_w, len_l = torch.randint(5,20,(B,)), torch.randint(5,20,(B,))
lp_pw = torch.randn(B); lp_pl = torch.randn(B)
s_loss, s_acc = simpo_loss(lp_pw, lp_pl, len_w, len_l)
assert s_loss.shape == ()

is_good = torch.tensor([True,True,False,True,False,False,True,False])
kto = kto_loss(torch.randn(B), torch.randn(B), is_good)
assert kto.shape == ()
print(f"ipo_loss ✓  simpo_loss ✓  kto_loss ✓")


---
## 3 · GRPO vs PPO — Deep Comparison

Both use the PPO clipped objective, but they differ fundamentally in how advantages are computed.

### PPO (online, with critic)
```
For each batch:
  1. Collect rollouts with π_θ
  2. Compute V(s_t) using value network V_φ
  3. Compute GAE: δ_t = r_t + γV(s_{t+1}) - V(s_t), Â_t = Σ (γλ)^k δ_{t+k}
  4. Update π_θ with clipped surrogate
  5. Update V_φ with MSE loss on returns
```

### GRPO (online, no critic)
```
For each batch of prompts:
  1. Sample G completions per prompt
  2. Score each with reward function → R_i
  3. Normalise within group: A_i = (R_i - mean_G) / std_G
  4. Update π_θ with clipped surrogate using A_i
```

### Key differences

| Aspect | PPO | GRPO |
|---|---|---|
| Value network | Yes (same backbone, extra head) | **No** |
| Advantage estimator | GAE (bootstrapped) | Group mean (MC within group) |
| Memory | 2× model memory | 1× model memory |
| Bias | Lower (bootstrapping) | Higher (MC) |
| Variance | Higher (bootstrap noise) | Lower (group normalisation) |
| Sample efficiency | Higher (reuse rollouts) | Lower (fresh samples per prompt) |
| Stability | Requires careful V_φ tuning | More robust, fewer hyperparams |

### When to use each
- **PPO:** complex tasks with dense per-step rewards (games, robotics, multi-turn dialogue)
- **GRPO:** terminal reward tasks (math correctness, coding, summarisation) — especially at scale


In [ ]:
def ppo_advantage_with_critic(
    rewards:     torch.Tensor,  # (B, T)
    values:      torch.Tensor,  # (B, T)  detached
    next_values: torch.Tensor,  # (B, T)  detached
    comp_mask:   torch.Tensor,  # (B, T)
    gamma: float = 0.99,
    lam:   float = 0.95,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Token-level GAE for LLM PPO.
    Backward loop over t:
      delta = rewards[:,t] + gamma*next_values[:,t] - values[:,t]
      gae   = (delta + gamma*lam*gae) * comp_mask[:,t]
      adv[:,t] = gae
      returns[:,t] = adv[:,t] + values[:,t]
    Returns (advantages, returns).
    """
    # TODO
    raise NotImplementedError


def grpo_advantage(rewards: torch.Tensor, G: int, eps: float = 1e-8) -> torch.Tensor:
    """Group-relative normalisation. Reshape to (-1, G), normalise per row, reshape back."""
    # TODO
    raise NotImplementedError


def compare_advantage_variance(G: int = 8, n_trials: int = 1000) -> Dict[str, float]:
    """
    Simulate rewards with varying group means; compare raw_reward_std vs grpo_adv_std.
    grpo_adv_std should be much smaller (close to 1.0 regardless of raw scale).
    """
    # TODO: simulate n_trials prompts, each with G completions, random group means
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, G = 4, 12, 4
rewards   = torch.randn(B, T) * 0.1
values    = torch.randn(B, T)
nxt_vals  = torch.cat([values[:, 1:].detach(), torch.zeros(B, 1)], dim=1)
comp_mask = torch.cat([torch.zeros(B, 4), torch.ones(B, 8)], dim=1).long()

adv, rets = ppo_advantage_with_critic(rewards, values.detach(), nxt_vals.detach(), comp_mask)
assert adv.shape == (B, T) and rets.shape == (B, T)
assert (adv[:, :4] == 0).all(), "Prompt positions must be zero"

grpo_adv = grpo_advantage(torch.randn(B*G), G=G)
for i in range(B):
    assert grpo_adv[i*G:(i+1)*G].mean().abs() < 1e-4, f"Group {i} mean not zero"

stats = compare_advantage_variance(G=8)
print(f"ppo_advantage_with_critic ✓  grpo_advantage ✓")
print(f"  raw std={stats['raw_reward_std']:.3f}  grpo std={stats['grpo_adv_std']:.3f}")


---
## 4 · Offline vs Online — Distribution Shift

### Offline methods (DPO, IPO, SimPO)
- Train on a **fixed dataset** of human preferences
- No new generations during training
- Risk: **distribution shift** — the policy may move far from the data-generating distribution,  
  making the preference labels unreliable (they were collected under a different model)
- Mitigation: **iterative DPO** — periodically generate new completions and collect new labels

### Online methods (PPO, GRPO)
- Continuously generate new completions and score them
- Always on-policy (or near-policy with clipping)
- More expensive but avoids distribution shift by design
- **On-policy ratio:** how close are current rollouts to the training distribution?

### Measuring distribution shift
The probability ratio $\rho = \pi_\theta(y|x) / \pi_{data}(y|x)$ measures how much the current policy  
differs from the data-generating policy. PPO clips this; DPO has no such mechanism.

### The offline-online spectrum

| Method | Data | Updates | Stability | Performance |
|---|---|---|---|---|
| SFT | Static | Offline | High | Baseline |
| DPO | Static preference pairs | Offline | High | Good |
| Iterative DPO | Refreshed periodically | Semi-online | Medium | Better |
| GRPO | Generated on-the-fly | Online | Medium | Strong |
| PPO | Generated on-the-fly | Online | Lower | Strongest |


In [ ]:
def importance_weight(
    logprobs_policy:    torch.Tensor,  # (B,)
    logprobs_behaviour: torch.Tensor,  # (B,)  detached
    clip_ratio: float = 5.0,
) -> torch.Tensor:
    """
    ρ = exp(log π_θ - log π_data), clipped to [1/clip_ratio, clip_ratio].
    """
    # TODO
    raise NotImplementedError


def dpo_with_sft_regularizer(
    logprobs_policy_chosen:   torch.Tensor,  # (B,)
    logprobs_policy_rejected: torch.Tensor,  # (B,)
    logprobs_ref_chosen:      torch.Tensor,  # (B,)  detached
    logprobs_ref_rejected:    torch.Tensor,  # (B,)  detached
    beta:     float = 0.1,
    sft_coef: float = 0.1,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    L = L_DPO + sft_coef * (-mean(logp_policy_chosen))
    Return (total_loss, dict with dpo/sft/total).
    """
    # TODO
    raise NotImplementedError


def estimate_distribution_shift(
    logprobs_current: torch.Tensor,  # (B,)
    logprobs_data:    torch.Tensor,  # (B,)  detached
) -> Dict[str, float]:
    """
    Compute:
      log_rho = logp_current - logp_data
      rho     = exp(log_rho)
      ess     = (Σρ)² / Σρ²
    Return dict with mean_log_ratio, max_log_ratio, ess, ess_fraction (ess/B).
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
lp_data = torch.randn(B)
iw_same = importance_weight(lp_data, lp_data)
assert abs(iw_same.mean().item() - 1.0) < 1e-4

s_same  = estimate_distribution_shift(lp_data, lp_data)
assert abs(s_same["ess_fraction"] - 1.0) < 1e-4
s_large = estimate_distribution_shift(lp_data + 3.0, lp_data)
assert s_large["ess_fraction"] < 0.5

lp_pw = torch.randn(B, requires_grad=True)
lp_pl = torch.randn(B)
total, info = dpo_with_sft_regularizer(lp_pw, lp_pl, lp_pw.detach(), lp_pl.detach())
total.backward()
assert lp_pw.grad is not None
print(f"importance_weight ✓  ESS same={s_same['ess_fraction']:.3f}  large={s_large['ess_fraction']:.3f}")
print(f"dpo_with_sft_regularizer ✓  {info}")


---
## 5 · Algorithm Selection & Practical Comparison

Putting it all together — when to use which algorithm.

### Decision tree
```
Do you have a verifiable reward function (math, code)?
  Yes → GRPO or PPO (online RL)
  No  → Do you have preference pairs?
          Yes → DPO family (offline)
                  Are you worried about overfitting / reward margin?
                    Yes → IPO
                  Is your data unpaired?
                    Yes → KTO
                  Do you have no reference model?
                    Yes → SimPO
          No  → Need to collect preference data first
```

### Key hyperparameters and their effects
| Hyperparameter | Too low | Too high |
|---|---|---|
| β (KL weight) | Reward hacking | Too conservative, no learning |
| ε (PPO clip) | Under-update | Large policy updates, instability |
| G (GRPO groups) | High variance advantages | Expensive (G × forward passes) |
| γ (discount) | Ignores future | Slow convergence, instability |
| λ (GAE) | High bias | High variance |


In [ ]:
def run_algorithm_comparison(
    n_steps: int = 50,
    B: int = 16,
    G: int = 4,
    seed: int = 42,
) -> Dict[str, List[float]]:
    """
    Toy comparison of DPO vs GRPO dynamics.
    Both have a scalar theta parameter; simulate appropriate losses.

    DPO loop:
      - Sample features feat_w (positive), feat_l (negative)
      - lp_pol = theta * feat; lp_ref = 0
      - compute dpo_loss, step theta_dpo

    GRPO loop:
      - Sample B*G features; reward = theta * features + features
      - compute grpo_advantage; compute clipped PPO surrogate; step theta_grpo

    Track dpo_reward_margin and grpo_reward over steps.
    """
    # TODO
    raise NotImplementedError


results = run_algorithm_comparison(n_steps=50)
assert results['final_theta_dpo'] > 0, "DPO should push theta positive (toward chosen)"
assert results['dpo_reward_margin'][-1] > results['dpo_reward_margin'][0], "Margin should grow"
print("Algorithm comparison ✓")
print(f"  DPO  final theta={results['final_theta_dpo']:.4f}  margin={results['dpo_reward_margin'][-1]:.4f}")
print(f"  GRPO final theta={results['final_theta_grpo']:.4f}  reward={results['grpo_reward'][-1]:.4f}")
